# RAG Mental Model and Architecture

| Field | Value |
|---|---|
| Stage | Foundations |
| Difficulty | Beginner |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-24 |

Callout - Key idea:
RAG is two connected systems: an offline path that prepares trustworthy knowledge and an online path that retrieves evidence before generating an answer.

## 30-Second Summary

Retrieval-Augmented Generation gives a generator selected external evidence at query time. The retriever decides **what the model is allowed to see**; the generator decides **how to answer from it**. Better prompting cannot repair missing evidence, and perfect retrieval cannot guarantee a faithful answer. Measure both stages separately.

## Why This Matters

Imagine a Northstar customer asking when deleted documents disappear from search. The answer lives in a changing operations policy, not safely inside a model's parameters. A useful system must find the correct policy, preserve its identity, place it in the prompt, answer from it, and cite it. Every arrow can fail.

## Scope

| Covers | Does not cover |
|---|---|
| Indexing path, query path, grounding, citations, evaluation boundaries | Chunking algorithms, embedding models, vector-index internals, agent loops |

### Prerequisites

Basic Python is helpful. No model download or API key is required.

## Mental Model

```text
OFFLINE: sources -> parse -> clean -> chunk -> represent -> index
                                                    |
ONLINE:  question -> transform -> retrieve -> select context -> generate
                                      |                    |
                                  source IDs          cited answer
                                      \____________________/
                                           evaluation
```

The index is external, updateable memory. Retrieval narrows that memory into a small evidence set. Generation should be conditioned on that evidence and allowed to abstain when it is insufficient. Citations connect output claims back to inspectable source IDs.

The production boundary is wider than the model call. Source freshness, permissions, document identity, retrieval quality, context limits, and answer grounding all affect the final result.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from rag_101 import TfidfRetriever, extractive_answer, load_corpus

documents = load_corpus()
len(documents), documents[0].id

## How It Works

| Stage | Input | Responsibility | Typical failure |
|---|---|---|---|
| Parse and clean | Source files/systems | Preserve useful content and metadata | Lost tables, OCR noise, missing ownership |
| Chunk and represent | Documents | Create retrievable units | Split facts, chunks too broad, domain mismatch |
| Index | Vectors/terms plus metadata | Store searchable, permission-aware knowledge | Stale entries, missing deletes, cross-tenant exposure |
| Retrieve and select | User question | Return relevant, diverse, allowed evidence | Relevant source absent or buried |
| Generate | Question plus context | Answer only from adequate evidence | Unsupported claim or ignored instruction |
| Evaluate | Trace plus expected behavior | Locate the failing stage | One end-to-end score hides the cause |

The most useful debugging question is not “Why was the answer bad?” It is “At which boundary did the expected evidence or behavior disappear?”

## Baseline

A parametric-only answer has no visible connection to the repository's current policies. Our baseline therefore uses the smallest observable RAG path: retrieve documents, select one supporting sentence, attach its document ID, or abstain. The extractive generator is intentionally simple so retrieval and grounding remain visible.

In [ ]:
retriever = TfidfRetriever(documents)
question = 'What happens to the search index when a source document is deleted?'
retrieved = retriever.search(question, k=3)

[(item.document.id, round(item.score, 3)) for item in retrieved]

## Technique Implementation

A trace should expose the question, ranked source IDs, scores, chosen citation, answer, and abstention decision. Production traces add timings, model/index versions, tenant/permission context, and errors—but should not indiscriminately copy sensitive content.

In [ ]:
def run_observable_rag(question: str, k: int = 3) -> dict:
    results = retriever.search(question, k=k)
    generated = extractive_answer(question, results)
    return {
        'question': question,
        'retrieval': [
            {'document_id': item.document.id, 'score': round(item.score, 4)}
            for item in results
        ],
        **generated,
    }


run_observable_rag(question)

## Controlled Experiment

Compare a supported question with an unsupported one. The pipeline and threshold stay fixed; only evidence availability changes. A healthy system answers the first with a citation and abstains on the second.

In [ ]:
supported = run_observable_rag(question)
unsupported = run_observable_rag('What is the minimum password length for Northstar accounts?')

{
    'supported': {key: supported[key] for key in ('answer', 'citation', 'abstained')},
    'unsupported': {key: unsupported[key] for key in ('answer', 'citation', 'abstained')},
}

## Evaluation

| Layer | Ask | Example signal |
|---|---|---|
| Retrieval | Did the expected evidence appear early? | Hit rate/recall@k, MRR |
| Context | Was evidence preserved, allowed, and within budget? | Source coverage, permission violations, tokens |
| Generation | Is the answer supported and useful? | Groundedness, answer quality, citation accuracy |
| Operations | Is behavior affordable and reliable? | p50/p95 latency, errors, tokens/cost, freshness |

End-to-end answer quality matters, but stage-level measures tell us what to fix.

In [ ]:
assert supported['citation'] == 'northstar-indexing'
assert supported['abstained'] is False
assert unsupported['citation'] is None
assert unsupported['abstained'] is True
print('Mental-model checks passed.')

## Decision Guide

| Situation | Choose | Reason | Trade-off |
|---|---|---|---|
| Current private knowledge with source attribution | Two-step RAG | Predictable retrieval before generation | Index and evaluation work |
| Stable rules that fit safely in a prompt | Prompt/context configuration | Fewer moving parts | Updates require prompt/config changes |
| Exact structured calculations | Database/API tool | Preserves schema and deterministic operations | Requires tool contract and authorization |
| Model must decide among multiple knowledge tools | Agentic retrieval | Flexible routing | Variable latency, cost, and failure paths |

Do not add agents merely because the basic retriever is weak. Fix the measured retrieval failure first.

## Failure Modes and Debugging

| Symptom | Likely cause | How to verify | Fix |
|---|---|---|---|
| Correct document never appears | Parsing/chunking/representation/query mismatch | Inspect expected source rank | Repair the earliest failing stage |
| Correct document appears but answer is wrong | Context assembly or generation failure | Compare retrieved text with prompt and claim | Improve instructions, selection, or model |
| Old answer persists after update | Index lifecycle/cache invalidation failure | Trace source and index versions | Re-index or invalidate stale entries |
| Answer cites another tenant's source | Authorization applied after retrieval | Inspect pre-ranking candidate set | Filter by access before ranking |
| System answers unsupported question | Missing abstention policy | Test unanswerable golden questions | Add evidence threshold and refusal behavior |

## Production Notes

### Observability
Record stage timings, query/index versions, ranked IDs and scores, selected sources, model configuration, and outcome labels. Redact sensitive content.

### Safety and Guardrails
Apply tenant and user authorization before ranking. Treat retrieved text as untrusted data, not system instructions. Preserve source ownership and deletion requirements.

### Latency and Cost
Indexing cost happens when sources change; retrieval and generation cost happen per query. More candidates and model calls may improve quality but increase latency and cost.

## Practice

Draw the offline and online paths from memory. Then place these failures on the correct arrow: stale policy, missing citation, irrelevant top result, cross-tenant chunk, unsupported answer, and slow model call.

## Recall

Toggle - Recall: What problem does RAG solve?
It supplies selected, updateable external evidence to generation at query time.

Toggle - Recall: Why measure retrieval and generation separately?
Because a bad answer can come from missing evidence or from mishandling good evidence, and the fixes differ.

Toggle - Recall: What is the grounding boundary?
The answer's factual claims should be supported by the context selected from allowed sources.

Toggle - Recall: When should the system abstain?
When retrieved evidence is missing, insufficient, conflicting, or unauthorized.

## Sources

- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [LangChain retrieval documentation](https://docs.langchain.com/oss/python/deepagents/retrieval)

The Northstar policies are fictional repository-owned learning data.

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-24 | Complete | High | Revisit after the evaluation module adds deeper metrics |